# Correlation Engine

Loads combined sentiment from `sentiment_outputs/` and stock prices,
then runs the full correlation analysis.

## Key differences from previous approach
- Uses all tickers with signal (500+) not just 24
- Finds key price movements FIRST, then looks for preceding sentiment
- Cross-ticker spillover with rolling windows
- Source attribution: which source triggered each signal

## Outputs saved to GitHub
| File | Contents |
|------|----------|
| `correlation_outputs/corr_matrix.csv` | Full correlation matrix |
| `correlation_outputs/key_moves.csv` | Big price moves + preceding sentiment |
| `correlation_outputs/best_per_ticker.csv` | Best signal per ticker |
| `correlation_outputs/spillover_pairs.csv` | Cross-ticker pairs |
| `correlation_outputs/source_attribution.csv` | Which source drives each ticker |

---
## 0. Install

In [1]:
# !pip install pandas requests python-dotenv scipy

---
## 1. Configuration & Load

In [2]:
import os, io, requests, warnings
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO      = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN     = os.environ.get('GITHUB_TOKEN', None)
OUTPUT_PREFIX    = 'sentiment_outputs'
CORR_PREFIX      = 'correlation_outputs'
STOCKS_PREFIX    = 'stocks'

ANALYSIS_START   = '2015-01-01'
ANALYSIS_END     = datetime.now(timezone.utc).strftime('%Y-%m-%d')
SENTIMENT_THRESHOLD = 0.05
ZSCORE_THRESHOLD    = 2.0
MIN_SIGNAL_DAYS     = 10
PRICE_HORIZONS      = [1, 2, 3, 5, 7, 10, 21]
SPILLOVER_WINDOW    = 90

def load_csv(path, token=None):
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404: raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content: return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

def push_csv(df, path, token, msg=None):
    import base64
    GITHUB_API = 'https://api.github.com'
    if msg is None: msg = f'Update {path} - {len(df):,} rows'
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = {'Authorization': f'Bearer {token}',
               'Accept': 'application/vnd.github+json',
               'X-GitHub-Api-Version': '2022-11-28'}
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha: payload['sha'] = sha
    import time
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'  Saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha: payload['sha'] = sha
        else:
            print(f'  FAILED: {resp.status_code}')
            return False

print('Loading sentiment signals (quarterly files) ...')
sig_frames = []
for year in range(int(ANALYSIS_START[:4]), int(ANALYSIS_END[:4])+1):
    for q in [1, 2, 3, 4]:
        try:
            df_y = load_csv(f'{OUTPUT_PREFIX}/daily_signals_{year}_Q{q}.csv', GITHUB_TOKEN)
            sig_frames.append(df_y)
            print(f'  {year} Q{q}: {len(df_y):,} rows, {df_y["ticker"].nunique()} tickers')
        except FileNotFoundError:
            pass
# Fallback: also try yearly files (for compatibility)
if not sig_frames:
    for year in range(int(ANALYSIS_START[:4]), int(ANALYSIS_END[:4])+1):
        try:
            df_y = load_csv(f'{OUTPUT_PREFIX}/daily_signals_{year}.csv', GITHUB_TOKEN)
            sig_frames.append(df_y)
            print(f'  {year}: {len(df_y):,} rows (yearly file)')
        except FileNotFoundError:
            pass

if not sig_frames:
    raise RuntimeError('No sentiment data found. Run A_sentiment_engine.ipynb first.')

daily_signals = pd.concat(sig_frames, ignore_index=True)
daily_signals['date'] = pd.to_datetime(daily_signals['date'])
all_tickers = daily_signals['ticker'].unique().tolist()
print(f'\nLoaded: {len(daily_signals):,} ticker-days, {len(all_tickers):,} tickers')

Loading sentiment signals (quarterly files) ...
  2015 Q1: 197,010 rows, 2189 tickers
  2015 Q2: 199,199 rows, 2189 tickers
  2015 Q3: 201,388 rows, 2189 tickers
  2015 Q4: 201,388 rows, 2189 tickers
  2016 Q1: 199,199 rows, 2189 tickers
  2016 Q2: 199,199 rows, 2189 tickers
  2016 Q3: 201,388 rows, 2189 tickers
  2016 Q4: 201,388 rows, 2189 tickers
  2017 Q1: 197,010 rows, 2189 tickers
  2017 Q2: 199,199 rows, 2189 tickers
  2017 Q3: 201,388 rows, 2189 tickers
  2017 Q4: 201,388 rows, 2189 tickers
  2018 Q1: 197,010 rows, 2189 tickers
  2018 Q2: 199,199 rows, 2189 tickers
  2018 Q3: 201,388 rows, 2189 tickers
  2018 Q4: 201,388 rows, 2189 tickers
  2019 Q1: 197,010 rows, 2189 tickers
  2019 Q2: 199,199 rows, 2189 tickers
  2019 Q3: 201,388 rows, 2189 tickers
  2019 Q4: 201,388 rows, 2189 tickers
  2020 Q1: 199,199 rows, 2189 tickers
  2020 Q2: 199,199 rows, 2189 tickers
  2020 Q3: 201,388 rows, 2189 tickers
  2020 Q4: 201,388 rows, 2189 tickers
  2021 Q1: 197,010 rows, 2189 tickers
  

---
## 2. Load Stock Prices for All Tickers

In [3]:
print('Loading stock prices ...')
price_frames = []
missing = []

for i, ticker in enumerate(all_tickers):
    if i % 100 == 0: print(f'  {i}/{len(all_tickers)} ...', end='\r')
    try:
        df_p = load_csv(f'{STOCKS_PREFIX}/prices_{ticker}.csv', GITHUB_TOKEN)
        if df_p.empty or 'Close' not in df_p.columns:
            missing.append(ticker)
            continue
        df_p['Date']         = pd.to_datetime(df_p['Date'])
        df_p['ticker']       = ticker
        df_p['daily_return'] = df_p['Close'].pct_change(fill_method=None)
        df_p['roll_mean']    = df_p['daily_return'].rolling(60, min_periods=20).mean()
        df_p['roll_std']     = df_p['daily_return'].rolling(60, min_periods=20).std()
        df_p['zscore']       = ((df_p['daily_return'] - df_p['roll_mean']) / df_p['roll_std'])
        df_p['is_abnormal']  = df_p['zscore'].abs() >= ZSCORE_THRESHOLD
        price_frames.append(
            df_p[['Date','ticker','Close','daily_return','roll_std','zscore','is_abnormal']]
        )
    except FileNotFoundError:
        missing.append(ticker)

df_prices = pd.concat(price_frames, ignore_index=True) if price_frames else pd.DataFrame()
priced_tickers = df_prices['ticker'].unique().tolist() if not df_prices.empty else []
print(f'\nPrices loaded: {len(priced_tickers):,} tickers')
print(f'No price data: {len(missing):,} tickers')

Loading stock prices ...
  2100/2189 ...
Prices loaded: 2,186 tickers
No price data: 3 tickers


---
## 3. Find Key Price Movements First

Finds every date where a stock had an abnormal move (z-score > threshold),
then looks back 1-7 days to find what sentiment preceded it.
This is the correct order: price move → look back for sentiment.

In [4]:
print('Finding key price movements and preceding sentiment ...')

key_move_rows = []

sig_idx = daily_signals.set_index(['ticker','date'])

for ticker in priced_tickers:
    price = df_prices[df_prices['ticker']==ticker].set_index('Date').sort_index()
    big_moves = price[price['zscore'].abs() >= ZSCORE_THRESHOLD]

    for move_date, mrow in big_moves.iterrows():
        # Look back up to 7 days for sentiment
        for lookback in range(1, 8):
            sent_date = move_date - pd.Timedelta(days=lookback)
            try:
                srow = sig_idx.loc[(ticker, sent_date)]
                sent_val = srow.get('norm_sentiment', np.nan)
                if pd.isna(sent_val): continue
                # Check if sentiment direction matched move direction
                move_dir  = 'up' if mrow['daily_return'] > 0 else 'down'
                sent_dir  = 'positive' if sent_val >= SENTIMENT_THRESHOLD else (
                            'negative' if sent_val <= -SENTIMENT_THRESHOLD else 'neutral')
                predicted = ((sent_dir=='positive' and move_dir=='up') or
                             (sent_dir=='negative' and move_dir=='down'))
                key_move_rows.append({
                    'ticker'         : ticker,
                    'move_date'      : move_date,
                    'return_pct'     : round(mrow['daily_return']*100, 3),
                    'zscore'         : round(mrow['zscore'], 3),
                    'move_direction' : move_dir,
                    'sent_date'      : sent_date,
                    'days_before'    : lookback,
                    'sentiment'      : round(sent_val, 4),
                    'sent_direction' : sent_dir,
                    'predicted'      : predicted,
                    'story_count'    : srow.get('story_count', 0),
                    'sources_active' : srow.get('sources_active', ''),
                })
                break  # use closest preceding sentiment day
            except KeyError:
                continue

df_key_moves = pd.DataFrame(key_move_rows)
if not df_key_moves.empty:
    df_key_moves['move_date'] = pd.to_datetime(df_key_moves['move_date'])
    df_key_moves['sent_date'] = pd.to_datetime(df_key_moves['sent_date'])
    total       = len(df_key_moves)
    predicted   = df_key_moves['predicted'].sum()
    no_sent     = len(df_prices[df_prices['is_abnormal']]) - total
    print(f'Key moves found    : {total:,}')
    print(f'Sentiment preceded : {total:,} ({total/(total+no_sent):.1%} of all big moves)')
    print(f'Correctly predicted: {predicted:,} ({predicted/total:.1%})')
    print(f'No sentiment found : {no_sent:,} (blind spots)')
    print('\nTop 10 biggest predicted moves:')
    print(df_key_moves[df_key_moves['predicted']]
          .sort_values('zscore', key=abs, ascending=False)
          .head(10)[['ticker','move_date','return_pct','zscore',
                     'days_before','sentiment','sources_active']]
          .to_string(index=False))

Finding key price movements and preceding sentiment ...
Key moves found    : 22,270
Sentiment preceded : 22,270 (7.5% of all big moves)
Correctly predicted: 5,585 (25.1%)
No sentiment found : 274,105 (blind spots)

Top 10 biggest predicted moves:
ticker  move_date  return_pct  zscore  days_before  sentiment                                                               sources_active
  ALOY 2017-03-29       4.571   7.617            1     0.1018                                                                           hn
  ALOY 2019-01-04      81.455   7.617            1     0.6250                                      reddit_technology|reddit_wallstreetbets
  CAPS 2020-11-17       6.667   7.617            3     0.7390                                                                           hn
  TONX 2015-07-28      53.077   7.617            4     0.8361                                                             reddit_investing
  RDIB 2017-07-07       5.556   7.617            1     0.1

---
## 4. Full Correlation Matrix

In [5]:
print('Running correlation matrix (all tickers x all horizons) ...')
print('This may take 10-20 minutes for large universes.')

signal_cols = ['norm_sentiment','adaptive_sentiment','roll_3d','roll_5d',
               'roll_7d','roll_21d','sent_momentum','volume_surge']
signal_cols = [c for c in signal_cols
               if c in daily_signals.columns]

corr_results = []
tickers_with_both = [t for t in all_tickers if t in priced_tickers]
print(f'Running for {len(tickers_with_both):,} tickers with both signal and price data')

for i, ticker in enumerate(tickers_with_both):
    if i % 50 == 0: print(f'  {i}/{len(tickers_with_both)} ...', end='\r')
    sig   = daily_signals[daily_signals['ticker']==ticker].set_index('date').sort_index()
    price = df_prices[df_prices['ticker']==ticker].set_index('Date').sort_index()
    if len(sig) < MIN_SIGNAL_DAYS: continue

    for sig_col in signal_cols:
        if sig_col not in sig.columns: continue
        sent_series = sig[sig_col].dropna()
        if len(sent_series) < MIN_SIGNAL_DAYS: continue

        for horizon in PRICE_HORIZONS:
            pairs = []
            for date, sv in sent_series.items():
                future = price.index[price.index > date]
                if len(future) < horizon: continue
                ret = price.loc[future[horizon-1], 'daily_return']
                if not pd.isna(ret):
                    pairs.append((sv, ret))
            if len(pairs) < MIN_SIGNAL_DAYS: continue
            arr = np.array(pairs)
            r, p = stats.pearsonr(arr[:,0], arr[:,1])
            corr_results.append({
                'ticker'     : ticker,
                'signal'     : sig_col,
                'horizon'    : horizon,
                'corr'       : round(r, 4),
                'pval'       : round(p, 4),
                'n'          : len(pairs),
                'significant': p < 0.05,
            })

df_corr = pd.DataFrame(corr_results)
print(f'\nCorrelation matrix: {len(df_corr):,} combinations')

best_per_ticker = (
    df_corr.assign(abs_corr=df_corr['corr'].abs())
    .sort_values('abs_corr', ascending=False)
    .drop_duplicates(subset='ticker')
    .drop(columns='abs_corr')
    .sort_values('corr', key=abs, ascending=False)
    .reset_index(drop=True)
)
print('\nTop 20 tickers by absolute correlation:')
print(best_per_ticker.head(20)[['ticker','signal','horizon','corr','pval','n','significant']]
      .to_string(index=False))

Running correlation matrix (all tickers x all horizons) ...
This may take 10-20 minutes for large universes.
Running for 2,186 tickers with both signal and price data
  0/2186 ...

/var/folders/tv/j9n6pnp97znd2n0k2zmqv4jc0000gn/T/ipykernel_73265/1048007080.py:34: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(arr[:,0], arr[:,1])


  2150/2186 ...
Correlation matrix: 92,312 combinations

Top 20 tickers by absolute correlation:
ticker             signal  horizon    corr  pval  n  significant
  ACON            roll_5d        2 -1.0000   0.0 10         True
  MNDR            roll_5d       21  1.0000   0.0 12         True
 KELYB      sent_momentum        7 -1.0000   0.0 13         True
  TKLF            roll_5d        5 -0.9982   0.0 10         True
  DGNX           roll_21d        2  0.9980   0.0 42         True
  STRC           roll_21d        3 -0.9970   0.0 24         True
   SKK            roll_5d        2  0.9963   0.0 10         True
  GRNQ            roll_5d       21 -0.9963   0.0 10         True
  VRCA            roll_5d        3 -0.9959   0.0 10         True
  KYMR           roll_21d        2  0.9928   0.0 42         True
  EPSM            roll_5d       21 -0.9912   0.0 15         True
 GLIBA            roll_5d        2 -0.9888   0.0 15         True
  BIOA            roll_5d        2 -0.9875   0.0 10       

---
## 5. Source Attribution

For each ticker, which source contributed the most to the signal?

In [6]:
print('Computing source attribution ...')

try:
    df_raw = load_csv(f'{OUTPUT_PREFIX}/matched_items_raw.csv', GITHUB_TOKEN)
    df_raw['date'] = pd.to_datetime(df_raw['date'])

    source_attr = (
        df_raw.groupby(['ticker','source'])
        .agg(
            item_count    = ('sentiment',  'count'),
            avg_sentiment = ('sentiment',  'mean'),
            avg_confidence= ('confidence', 'mean'),
        )
        .reset_index()
    )
    # Find dominant source per ticker
    dominant = (
        source_attr.sort_values('item_count', ascending=False)
        .drop_duplicates(subset='ticker')
        .rename(columns={'source': 'dominant_source',
                          'item_count': 'dominant_count'})
    )
    print('Source breakdown across all tickers:')
    print(source_attr.groupby('source')['item_count'].sum()
          .sort_values(ascending=False).to_string())
    print('\nDominant source per ticker (sample):')
    print(dominant[['ticker','dominant_source','dominant_count']].head(20).to_string(index=False))
except FileNotFoundError:
    source_attr = pd.DataFrame()
    dominant    = pd.DataFrame()
    print('Raw items not found — run Notebook A first')

Computing source attribution ...
Raw items not found — run Notebook A first


---
## 6. Cross-Ticker Spillover (Rolling 90-Day Windows)

In [7]:
print('Computing cross-ticker spillover pairs ...')
print('Note: with 500+ tickers this is O(n^2) — sampling top 50 by signal strength')

MIN_CONSISTENCY = 0.30
MIN_AVG_CORR    = 0.12
TOP_N_TICKERS   = 50  # increase if you have time

# Use top tickers by signal strength for spillover analysis
top_by_signal = (
    daily_signals.groupby('ticker')['norm_sentiment']
    .apply(lambda x: x.notna().sum())
    .sort_values(ascending=False)
    .head(TOP_N_TICKERS)
    .index.tolist()
)
top_by_signal = [t for t in top_by_signal if t in priced_tickers]
print(f'Running spillover for top {len(top_by_signal)} tickers by signal coverage')

sent_wide = daily_signals[daily_signals['ticker'].isin(top_by_signal)].pivot_table(
    index='date', columns='ticker', values='norm_sentiment'
)
fwd_wide_rows = {}
for ticker in top_by_signal:
    price = df_prices[df_prices['ticker']==ticker].set_index('Date').sort_index()
    fwd = price['Close'].pct_change(5).shift(-5)
    fwd_wide_rows[ticker] = fwd
fwd_wide = pd.DataFrame(fwd_wide_rows)
fwd_wide.index = pd.to_datetime(fwd_wide.index)

spillover_rows = []
for st in top_by_signal:
    for pt in top_by_signal:
        if st == pt: continue
        if st not in sent_wide.columns or pt not in fwd_wide.columns: continue
        combined = pd.concat([
            sent_wide[st].rename('sent'),
            fwd_wide[pt].rename('ret')
        ], axis=1).dropna()
        if len(combined) < 60: continue
        roll = combined['sent'].rolling(SPILLOVER_WINDOW, min_periods=30).corr(combined['ret'])
        avg  = roll.mean()
        cons = max((roll > 0.2).mean(), (roll < -0.2).mean())
        if abs(avg) >= MIN_AVG_CORR and cons >= MIN_CONSISTENCY:
            spillover_rows.append({
                'sent_ticker' : st,
                'price_ticker': pt,
                'avg_corr'    : round(avg, 4),
                'consistency' : round(cons, 3),
                'direction'   : 'positive' if avg > 0 else 'negative',
            })

df_spillover = pd.DataFrame(spillover_rows).sort_values('avg_corr', key=abs, ascending=False)
print(f'Confirmed spillover pairs: {len(df_spillover)}')
if not df_spillover.empty:
    print(df_spillover.head(20).to_string(index=False))

Computing cross-ticker spillover pairs ...
Note: with 500+ tickers this is O(n^2) — sampling top 50 by signal strength
Running spillover for top 50 tickers by signal coverage


/var/folders/tv/j9n6pnp97znd2n0k2zmqv4jc0000gn/T/ipykernel_73265/2820254139.py:25: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = price['Close'].pct_change(5).shift(-5)


Confirmed spillover pairs: 17
sent_ticker price_ticker  avg_corr  consistency direction
       MIND         STRK    0.3518        0.628  positive
        AMD         PTRN    0.2929        0.603  positive
        CRM          FGL    0.2473        0.493  positive
       PLTR         PTRN   -0.2408        0.431  negative
       COIN         PTRN   -0.2197        0.411  negative
       NFLX         INTJ    0.2140        0.424  positive
       STRK         PTRN   -0.1992        0.458  negative
       SILC         PTRN    0.1963        0.302  positive
       AAPL         STRK   -0.1921        0.486  negative
       MRNA          FGL   -0.1912        0.397  negative
       NFLX          FGL   -0.1906        0.353  negative
        PRE         INTJ    0.1890        0.411  positive
        DLO         INTJ   -0.1742        0.361  negative
         XE          FGL    0.1669        0.321  positive
       MIND         PLTR    0.1519        0.357  positive
       SOGP         STHO    0.1422        

---
## 7. Save All Outputs

In [8]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    outputs = [
        (df_corr,        f'{CORR_PREFIX}/corr_matrix.csv',       'Full correlation matrix'),
        (best_per_ticker,f'{CORR_PREFIX}/best_per_ticker.csv',   'Best signal per ticker'),
        (df_key_moves,   f'{CORR_PREFIX}/key_moves.csv',         'Key moves with preceding sentiment'),
        (df_spillover,   f'{CORR_PREFIX}/spillover_pairs.csv',   'Cross-ticker spillover pairs'),
    ]
    if not source_attr.empty:
        outputs.append((source_attr, f'{CORR_PREFIX}/source_attribution.csv', 'Source attribution'))

    for df, path, desc in outputs:
        if df is not None and not df.empty:
            push_csv(df, path, GITHUB_TOKEN, desc)

    print('\nAll correlation outputs saved')

  Saved correlation_outputs/corr_matrix.csv (92,312 rows)
  Saved correlation_outputs/best_per_ticker.csv (2,184 rows)
  Saved correlation_outputs/key_moves.csv (22,270 rows)
  Saved correlation_outputs/spillover_pairs.csv (17 rows)

All correlation outputs saved
